# NeSy-RAG: attributable symbolic RAG with knowledge-gap detection

NeSy-RAG attaches a symbolic module to each retrieved text chunk, exposes module behavior through zero-arity Boolean predicates, composes those predicates into a query, and executes it with source-attributed traces. Dynamic predicates represent user facts that may be missing; symbolic evaluation identifies which missing values can change the answer and triggers targeted follow-up questions.

This notebook implements that information flow with deterministic Python. It requires no API key, model download, Prolog installation, or third-party package.

Paper: https://arxiv.org/abs/2608.06292

In [ ]:
# Retrieved domain chunks. Each symbolic module below has exactly one source chunk.
chunks = {
    "C1": (
        "A full-time student who lives more than 50 km from campus meets the "
        "study-status and distance conditions for the travel grant."
    ),
    "C2": (
        "A student whose travel is already funded by an employer is disqualified "
        "from receiving the travel grant."
    ),
    "C3": "The library reimbursement applies only to textbook purchases.",
}
question = "Can I receive the travel grant?"
print(question)

## 1. Chunk-attributed symbolic modules

The paper synthesizes Prolog from each chunk. Here each module exposes one zero-arity predicate whose requirements reference dynamic Boolean facts about the user.

In [ ]:
modules = {
    "M1": {
        "source_chunk_id": "C1",
        "predicates": {
            "meets_travel_grant_conditions": {
                "description": "meets travel grant study status and distance conditions",
                "all": [("is_full_time_student", True), ("lives_over_50km_from_campus", True)],
            }
        },
    },
    "M2": {
        "source_chunk_id": "C2",
        "predicates": {
            "is_disqualified_from_travel_grant": {
                "description": "is disqualified from travel grant because an employer funds travel",
                "all": [("has_employer_travel_funding", True)],
            }
        },
    },
    "M3": {
        "source_chunk_id": "C3",
        "predicates": {
            "can_claim_library_reimbursement": {
                "description": "can claim reimbursement for textbook purchases",
                "all": [("bought_textbooks", True)],
            }
        },
    },
}

dynamic_fact_questions = {
    "is_full_time_student": "Are you enrolled as a full-time student?",
    "lives_over_50km_from_campus": "Do you live more than 50 km from campus?",
    "has_employer_travel_funding": "Is your travel already funded by an employer?",
    "bought_textbooks": "Did you buy textbooks?",
}
print(f"Created {len(modules)} source-attributed modules.")

In [ ]:
# 2. Retrieve a small set of zero-arity predicates relevant to the question.
import re
from itertools import product

STOPWORDS = {"a", "an", "the", "i", "can", "could", "do", "from", "is", "to"}

def tokens(text):
    return {token for token in re.findall(r"[a-z0-9]+", text.casefold()) if token not in STOPWORDS}

predicate_catalog = []
for module_id, module in modules.items():
    for predicate, rule in module["predicates"].items():
        predicate_catalog.append({
            "predicate": predicate,
            "description": rule["description"],
            "module_id": module_id,
            "source_chunk_id": module["source_chunk_id"],
        })

def retrieve_predicates(question, top_k=2):
    query_tokens = tokens(question)
    ranked = []
    for item in predicate_catalog:
        score = len(query_tokens & tokens(item["description"]))
        ranked.append((score, item["predicate"], item))
    return [item for _, _, item in sorted(ranked, key=lambda row: (-row[0], row[1]))[:top_k]]

selected_predicates = retrieve_predicates(question)
for item in selected_predicates:
    print(item)
assert {item["predicate"] for item in selected_predicates} == {
    "meets_travel_grant_conditions", "is_disqualified_from_travel_grant"
}

## 3. Query composition and attributed execution

A full NeSy-RAG system asks an LLM to compose a Prolog query from the retrieved predicates. The equivalent Boolean query here is: `meets_travel_grant_conditions AND NOT is_disqualified_from_travel_grant`.

In [ ]:
query_expression = {
    "all": [
        ("meets_travel_grant_conditions", True),
        ("is_disqualified_from_travel_grant", False),
    ]
}

predicate_lookup = {}
for module_id, module in modules.items():
    for predicate, rule in module["predicates"].items():
        predicate_lookup[predicate] = (module_id, module["source_chunk_id"], rule)

def execute(context):
    trace = []
    predicate_results = {}
    for predicate, expected_query_value in query_expression["all"]:
        module_id, chunk_id, rule = predicate_lookup[predicate]
        checks = []
        for fact, expected_fact_value in rule["all"]:
            if fact not in context:
                raise KeyError(fact)
            checks.append({"fact": fact, "observed": context[fact], "expected": expected_fact_value})
        value = all(check["observed"] == check["expected"] for check in checks)
        predicate_results[predicate] = value
        trace.append({
            "predicate": predicate,
            "value": value,
            "module_id": module_id,
            "source_chunk_id": chunk_id,
            "fact_checks": checks,
        })
    answer = all(predicate_results[predicate] == expected for predicate, expected in query_expression["all"])
    return answer, trace

print("Symbolic query ready.")

In [ ]:
# 4. Knowledge-gap detection: test every assignment of missing facts.
def facts_used_by_query():
    used = []
    for predicate, _ in query_expression["all"]:
        _, _, rule = predicate_lookup[predicate]
        used.extend(fact for fact, _ in rule["all"])
    return sorted(set(used))

def possible_outcomes(partial_context):
    missing = [fact for fact in facts_used_by_query() if fact not in partial_context]
    rows = []
    for values in product([False, True], repeat=len(missing)):
        completed = dict(partial_context)
        completed.update(dict(zip(missing, values)))
        outcome, _ = execute(completed)
        rows.append({"assignment": dict(zip(missing, values)), "outcome": outcome})
    return missing, rows

def influential_missing_facts(partial_context):
    missing, rows = possible_outcomes(partial_context)
    influential = []
    for fact in missing:
        others = [item for item in missing if item != fact]
        grouped = {}
        for row in rows:
            key = tuple((other, row["assignment"][other]) for other in others)
            grouped.setdefault(key, {})[row["assignment"][fact]] = row["outcome"]
        if any(values.get(False) != values.get(True) for values in grouped.values() if len(values) == 2):
            influential.append(fact)
    return influential, rows

user_context = {
    "is_full_time_student": True,
    "lives_over_50km_from_campus": True,
    # has_employer_travel_funding is intentionally missing.
}
influential, assignment_results = influential_missing_facts(user_context)
print("Possible outcomes:", assignment_results)
print("Influential missing facts:", influential)
print("Follow-up:", dynamic_fact_questions[influential[0]])
assert influential == ["has_employer_travel_funding"]

In [ ]:
# 5. Incorporate the user's response and execute deterministically.
simulated_user_response = False
user_context[influential[0]] = simulated_user_response
answer, trace = execute(user_context)

print("Answer:", "Yes" if answer else "No")
print("Attributed execution trace:")
for step in trace:
    print(step)
    print("  Source:", chunks[step["source_chunk_id"]])

assert answer is True
assert {step["source_chunk_id"] for step in trace} == {"C1", "C2"}

## What the full method adds

The paper uses LLMs to synthesize Prolog modules from retrieved chunks and to compose queries from retrieved zero-arity predicates. Predicate retrieval uses joint natural-language/code embeddings, and execution runs in SWI-Prolog through PySwip. This notebook isolates the two most distinctive guarantees: every symbolic step retains a source-chunk address, and unresolved user facts are surfaced when their values can change the query result.